# 03 Dataset Creation

## Purpose

This notebook creates the machine learning dataset used for model training.

The notebook performs the following tasks:

1. Load UrbanSound8K metadata.
2. Locate all audio files.
3. Extract MFCC features.
4. Associate features with class labels.
5. Build a structured dataset.
6. Save the dataset for future model training.

Output:

ml/data/features/urban_sound_features.csv

In [16]:
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

from tqdm import tqdm

print("Libraries loaded successfully")

Libraries loaded successfully


## Dataset Configuration

Define paths to the UrbanSound8K dataset and metadata.

In [17]:
DATASET_PATH = Path("../data/raw/UrbanSound8K")
METADATA_PATH = DATASET_PATH / "UrbanSound8K.csv"

print("Dataset Exists:", DATASET_PATH.exists())
print("Metadata Exists:", METADATA_PATH.exists())

Dataset Exists: True
Metadata Exists: True


## Load Metadata

The metadata file contains the class labels and audio file locations.

In [19]:
metadata = pd.read_csv(METADATA_PATH)

print("Metadata Shape:", metadata.shape)

metadata.head()

Metadata Shape: (8732, 8)


,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


## Feature Extraction Function

This function extracts 40 MFCC coefficients and averages them across time to create a fixed-length feature vector.

In [20]:
def extract_features(file_path):

    try:

        audio, sr = librosa.load(
            file_path,
            sr=22050
        )

        mfccs = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=40
        )

        mfccs_scaled = np.mean(
            mfccs.T,
            axis=0
        )

        return mfccs_scaled

    except Exception as e:

        print("Error:", file_path)

        return None

## Generate Feature Dataset

Each audio file is converted into a feature vector and paired with its class label.

In [23]:
features_dataset = []

for _, row in tqdm(
    metadata.iterrows(),
    total=len(metadata)
):

    fold = row["fold"]

    filename = row["slice_file_name"]

    label = row["class"]

    audio_path = (
        DATASET_PATH /
        f"fold{fold}" /
        filename
    )

    features = extract_features(audio_path)

    if features is not None:

        feature_row = list(features)

        feature_row.append(label)

        features_dataset.append(feature_row)

  0%|          | 0/8732 [00:00<?, ?it/s]c:\Users\cyuba\Documents\LEARN\CAPSTONE\Noise\Urban-Noise-Governance-System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 41%|████      | 3552/8732 [01:07<01:35, 54.42it/s]c:\Users\cyuba\Documents\LEARN\CAPSTONE\Noise\Urban-Noise-Governance-System\.venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
 95%|█████████▌| 8321/8732 [02:31<00:06, 64.22it/s]c:\Users\cyuba\Documents\LEARN\CAPSTONE\Noise\Urban-Noise-Governance-System\.venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
 95%|█████████▌| 8329/8732 [02:31<00:06, 66.92it/s]c:\Users\cyuba\Documents\LEARN\CAPSTONE\Noise\Urban-Noise-Governance-S

## Create DataFrame

Convert extracted features into a structured machine learning dataset.

In [24]:
feature_columns = [
    f"mfcc_{i}"
    for i in range(40)
]

feature_columns.append("label")

features_df = pd.DataFrame(
    features_dataset,
    columns=feature_columns
)

features_df.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,...,mfcc_31,mfcc_32,mfcc_33,mfcc_34,mfcc_35,mfcc_36,mfcc_37,mfcc_38,mfcc_39,label
0,-211.936981,62.581207,-122.813148,-60.745293,-13.893760,-29.789835,-3.978875,11.682742,12.963828,8.336421,...,7.510886,-0.885360,2.393814,-6.392372,-2.131859,2.276015,-0.791084,-1.540068,1.321150,dog_bark
1,-417.005188,99.336624,-42.995586,51.073326,9.853778,7.969693,11.197088,1.929117,7.030397,4.270228,...,2.277220,-1.539243,0.767109,-0.878724,0.908738,-2.681854,1.706798,-1.793606,1.761385,children_playing
2,-452.393158,112.362534,-37.578068,43.195866,8.631845,15.379366,16.882149,1.233047,6.833122,3.900115,...,-0.154540,-5.285954,-0.790016,-2.979211,-0.202845,-3.088082,3.808014,-0.090056,0.869102,children_playing
3,-406.479218,91.196602,-25.043558,42.784519,11.586844,5.054164,12.431632,-1.599949,6.656064,1.442355,...,1.279155,0.635733,1.414131,-2.831120,1.808179,-2.178432,-0.436480,-3.051344,-0.170013,children_playing
4,-439.638733,103.862228,-42.658787,50.690277,12.209422,15.873466,11.729268,1.533585,11.292244,2.548622,...,-0.774089,-0.985246,2.566369,-2.976879,1.071741,-1.825602,3.291397,-0.169670,1.392584,children_playing


In [25]:
print("Dataset Shape:")

features_df.shape

Dataset Shape:


(8732, 41)

## Class Distribution

Verify that all UrbanSound8K classes are represented correctly.

In [26]:
features_df["label"].value_counts()

label
dog_bark            1000
children_playing    1000
air_conditioner     1000
street_music        1000
engine_idling       1000
jackhammer          1000
drilling            1000
siren                929
car_horn             429
gun_shot             374
Name: count, dtype: int64

## Save Feature Dataset

Persist extracted features for training and evaluation notebooks.

In [27]:
FEATURE_DIR = Path("../data/features")

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    FEATURE_DIR /
    "urban_sound_features.csv"
)

features_df.to_csv(
    output_file,
    index=False
)

print("Saved to:")
print(output_file)

Saved to:
..\data\features\urban_sound_features.csv


## Dataset Creation Summary

The UrbanSound8K dataset has been transformed into a machine-learning-ready feature dataset.

The generated file will be used by:

- 04_baseline_models.ipynb
- 05_cnn_model.ipynb
- 06_model_evaluation.ipynb

for training and evaluating acoustic event classification models.